In [ ]:
import math
from dataclasses import dataclass
from typing import Optional, Dict, Any
import sys, os
sys.path.append(os.path.abspath('../src/'))
from dataset import TrainDataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()



True

In [ ]:

# ---------------------------
# Utilities
# ---------------------------
class MLP(nn.Module):
    def __init__(self, d_in: int, d_hidden: int, d_out: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)


class SinCosPosEmb2D(nn.Module):
    """
    2D sin-cos positional embedding for grid (H,W).
    Returns (H*W, d_model).
    """
    def __init__(self, d_model: int, H: int, W: int):
        super().__init__()
        assert d_model % 4 == 0, "d_model must be divisible by 4 for 2D sin-cos"
        self.d = d_model
        self.H = H
        self.W = W

        pe = self._build(H, W, d_model)  # (HW, d)
        self.register_buffer("pe", pe, persistent=False)

    @staticmethod
    def _build(H, W, d):
        # allocate half for y, half for x; each half split into sin/cos
        d_half = d // 2
        d_quarter = d // 4
        y = torch.linspace(0, 1, steps=H)
        x = torch.linspace(0, 1, steps=W)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        yy = yy.reshape(-1, 1)  # (HW,1)
        xx = xx.reshape(-1, 1)

        freq = torch.exp(torch.linspace(0, math.log(10_000), steps=d_quarter))
        # (HW, d_quarter)
        y_sin = torch.sin(yy * freq)
        y_cos = torch.cos(yy * freq)
        x_sin = torch.sin(xx * freq)
        x_cos = torch.cos(xx * freq)

        pe = torch.cat([y_sin, y_cos, x_sin, x_cos], dim=-1)  # (HW, d)
        return pe.float()

    def forward(self):
        return self.pe  # (HW, d)


# ---------------------------
# Slot Attention (Locatello-style)
# ---------------------------
class SlotAttention(nn.Module):
    """
    Inputs:  x (B, N, D_in)  where N=H*W positions (for one time step)
    Outputs: slots (B, K, D_slot)
    """
    def __init__(
        self,
        num_slots: int,
        dim_in: int,
        dim_slot: int,
        iters: int = 3,
        eps: float = 1e-8,
        hidden_dim: int = 128,
    ):
        super().__init__()
        self.K = num_slots
        self.iters = iters
        self.eps = eps
        self.dim_slot = dim_slot

        self.norm_in = nn.LayerNorm(dim_in)
        self.norm_slots = nn.LayerNorm(dim_slot)
        self.norm_mlp = nn.LayerNorm(dim_slot)

        self.project_q = nn.Linear(dim_slot, dim_slot, bias=False)
        self.project_k = nn.Linear(dim_in, dim_slot, bias=False)
        self.project_v = nn.Linear(dim_in, dim_slot, bias=False)

        self.gru = nn.GRUCell(dim_slot, dim_slot)
        self.mlp = nn.Sequential(
            nn.Linear(dim_slot, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, dim_slot),
        )

        # learned Gaussian init for slots
        self.slots_mu = nn.Parameter(torch.zeros(1, 1, dim_slot))
        self.slots_logsigma = nn.Parameter(torch.zeros(1, 1, dim_slot))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, N, D_in)
        """
        B, N, _ = x.shape
        x = self.norm_in(x)

        k = self.project_k(x)  # (B,N,D)
        v = self.project_v(x)  # (B,N,D)

        # init slots
        mu = self.slots_mu.expand(B, self.K, -1)
        sigma = self.slots_logsigma.exp().expand(B, self.K, -1)
        slots = mu + sigma * torch.randn_like(mu)

        for _ in range(self.iters):
            slots_prev = slots
            slots_norm = self.norm_slots(slots)
            q = self.project_q(slots_norm)  # (B,K,D)

            # attention: dots between k and q
            # attn_logits: (B, N, K)
            attn_logits = torch.einsum("bnd,bkd->bnk", k, q) / math.sqrt(self.dim_slot)
            attn = F.softmax(attn_logits, dim=-1)  # over slots K
            # normalize over inputs N for each slot
            attn = attn + self.eps
            attn = attn / attn.sum(dim=1, keepdim=True)  # (B,N,K)

            # weighted sum of values -> updates (B,K,D)
            updates = torch.einsum("bnk,bnd->bkd", attn, v)

            # GRU expects (B*K, D)
            slots = self.gru(
                updates.reshape(B * self.K, -1),
                slots_prev.reshape(B * self.K, -1),
            ).reshape(B, self.K, -1)

            slots = slots + self.mlp(self.norm_mlp(slots))

        return slots  # (B,K,D_slot)


# ---------------------------
# Cross-attention Decoder (grid queries attend to slots)
# ---------------------------
class GridSlotDecoder(nn.Module):
    """
    Given slots for one time step (B,K,D), produce logits over vocab for each grid position (H*W).
    """
    def __init__(self, dim_slot: int, dim_q: int, vocab_size: int, H: int = 32, W: int = 32):
        super().__init__()
        self.H, self.W = H, W
        self.N = H * W
        self.vocab_size = vocab_size

        self.pos = SinCosPosEmb2D(dim_q, H, W)  # (N, dim_q)
        self.q_proj = nn.Linear(dim_q, dim_q, bias=False)
        self.k_proj = nn.Linear(dim_slot, dim_q, bias=False)
        self.v_proj = nn.Linear(dim_slot, dim_q, bias=False)

        self.out = nn.Sequential(
            nn.LayerNorm(dim_q),
            nn.Linear(dim_q, vocab_size),
        )

    def forward(self, slots: torch.Tensor) -> torch.Tensor:
        """
        slots: (B,K,D_slot)
        returns logits: (B, N, vocab)
        """
        B, K, _ = slots.shape
        q = self.pos().unsqueeze(0).expand(B, -1, -1)  # (B,N,dim_q)
        q = self.q_proj(q)

        k = self.k_proj(slots)  # (B,K,dim_q)
        v = self.v_proj(slots)  # (B,K,dim_q)

        # attn logits: (B,N,K)
        attn_logits = torch.einsum("bnc,bkc->bnk", q, k) / math.sqrt(q.shape[-1])
        attn = F.softmax(attn_logits, dim=-1)
        ctx = torch.einsum("bnk,bkc->bnc", attn, v)  # (B,N,dim_q)

        logits = self.out(ctx)  # (B,N,vocab)
        return logits


# ---------------------------
# State Encoder: (34,25) -> (6, d_state)
#  - past17 -> 3
#  - fut17  -> 3
# ---------------------------
class StateEncoder(nn.Module):
    def __init__(self, state_dim: int = 25, d_state: int = 128, dropout: float = 0.0):
        super().__init__()
        self.state_dim = state_dim
        self.d_state = d_state

        # encode per raw-frame
        self.frame_mlp = nn.Sequential(
            nn.LayerNorm(state_dim),
            nn.Linear(state_dim, d_state),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_state, d_state),
        )

        # compress 17 -> 3 using 1D conv with stride-ish (simple & stable)
        # We'll do: reshape to (B, d_state, 17) and use conv to 3 timesteps.
        self.conv17_to_3 = nn.Conv1d(d_state, d_state, kernel_size=5, stride=6, padding=2)  # 17 -> 3 approx

        # small post
        self.post = nn.Sequential(
            nn.LayerNorm(d_state),
            nn.Linear(d_state, d_state),
            nn.ReLU(inplace=True),
        )

    def forward(self, states_34: torch.Tensor) -> torch.Tensor:
        """
        states_34: (B, 34, 25) float
        returns:   (B, 6, d_state)
        """
        B, T, D = states_34.shape
        assert T == 34 and D == self.state_dim

        x = self.frame_mlp(states_34)  # (B,34,d_state)
        past = x[:, :17]               # (B,17,d)
        fut  = x[:, 17:]               # (B,17,d)

        def compress17(seq17):
            # (B,17,d) -> (B,d,17)
            z = seq17.transpose(1, 2)
            z = self.conv17_to_3(z)    # (B,d,3) because 17 -> 3 with stride 6
            z = z.transpose(1, 2)      # (B,3,d)
            return z

        past3 = compress17(past)
        fut3  = compress17(fut)
        out = torch.cat([past3, fut3], dim=1)  # (B,6,d_state)
        out = self.post(out)
        return out


# ---------------------------
# Token Encoder: tokens (B,3,32,32) -> features (B,3,N,d_in)
# ---------------------------
class TokenEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 256, H: int = 32, W: int = 32, dropout: float = 0.0):
        super().__init__()
        self.H, self.W = H, W
        self.N = H * W
        self.d = d_model

        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos2d = SinCosPosEmb2D(d_model, H, W)  # (N,d)
        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tok_3hw: torch.Tensor) -> torch.Tensor:
        """
        tok_3hw: (B, 3, H, W) int64
        returns: feats (B, 3, N, d)
        """
        B, T, H, W = tok_3hw.shape
        assert T == 3 and H == self.H and W == self.W

        x = self.embed(tok_3hw)  # (B,3,H,W,d)
        x = x.view(B, T, self.N, self.d)  # (B,3,N,d)

        pos = self.pos2d().unsqueeze(0).unsqueeze(0)  # (1,1,N,d)
        x = x + pos
        x = self.mlp(x)
        return x  # (B,3,N,d)


# ---------------------------
# Slot Dynamics: predict future slots using state sequence
# ---------------------------
class SlotDynamics(nn.Module):
    """
    slot-wise GRU conditioned on state at each of 6 steps.
    We:
      - infer slots for past 3 frames: S0,S1,S2
      - roll forward to produce S3,S4,S5 using states u3,u4,u5
    """
    def __init__(self, dim_slot: int, d_state: int):
        super().__init__()
        self.dim_slot = dim_slot
        self.d_state = d_state

        self.state_proj = nn.Linear(d_state, dim_slot)
        self.gru = nn.GRUCell(dim_slot, dim_slot)

        # optional slot interaction (light)
        self.slot_self_attn = nn.MultiheadAttention(embed_dim=dim_slot, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(dim_slot)

    def forward(self, slots_past: torch.Tensor, state_seq6: torch.Tensor) -> torch.Tensor:
        """
        slots_past:  (B,3,K,D)
        state_seq6:  (B,6,d_state)
        returns:     slots_future (B,3,K,D)  corresponding to t=3,4,5
        """
        B, T, K, D = slots_past.shape
        assert T == 3

        # start from last past slot
        s = slots_past[:, -1]  # (B,K,D)

        outs = []
        for step in range(3, 6):
            u = self.state_proj(state_seq6[:, step])  # (B,D)
            # broadcast to slots
            u = u.unsqueeze(1).expand(-1, K, -1)      # (B,K,D)

            # slot-wise GRU update
            s = self.gru(u.reshape(B*K, D), s.reshape(B*K, D)).reshape(B, K, D)

            # slot interaction
            attn_out, _ = self.slot_self_attn(s, s, s)
            s = self.norm(s + attn_out)

            outs.append(s)

        return torch.stack(outs, dim=1)  # (B,3,K,D)


# ---------------------------
# Full Model
# ---------------------------
@dataclass
class ModelConfig:
    vocab_size: int
    num_slots: int = 8
    d_token: int = 256
    d_slot: int = 256
    d_state: int = 128
    slot_iters: int = 3
    H: int = 32
    W: int = 32


class SlotPredictor(nn.Module):
    """
    Input:
      - past_tokens: (B,3,32,32) int64
      - robot_states: (B,34,25) float32
    Output:
      - logits_future: (B,3,32,32,vocab)  (or flattened)
    """
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg

        self.token_enc = TokenEncoder(cfg.vocab_size, cfg.d_token, cfg.H, cfg.W)
        self.slot_attn = SlotAttention(
            num_slots=cfg.num_slots,
            dim_in=cfg.d_token,
            dim_slot=cfg.d_slot,
            iters=cfg.slot_iters,
            hidden_dim=max(128, cfg.d_slot),
        )

        self.state_enc = StateEncoder(state_dim=25, d_state=cfg.d_state)

        # slots per frame: apply slot attention per time step
        self.in_proj = nn.Linear(cfg.d_token, cfg.d_token)  # optional

        self.dynamics = SlotDynamics(dim_slot=cfg.d_slot, d_state=cfg.d_state)

        self.decoder = GridSlotDecoder(
            dim_slot=cfg.d_slot,
            dim_q=cfg.d_slot,
            vocab_size=cfg.vocab_size,
            H=cfg.H, W=cfg.W
        )

    def forward(self, past_tokens_3hw: torch.Tensor, robot_states_34x25: torch.Tensor) -> torch.Tensor:
        """
        past_tokens_3hw: (B,3,32,32) long
        robot_states_34x25: (B,34,25) float
        returns logits: (B,3,HW,V)
        """
        B = past_tokens_3hw.shape[0]
        H, W = self.cfg.H, self.cfg.W
        N = H * W

        feats = self.token_enc(past_tokens_3hw)  # (B,3,N,d_token)

        # slot inference per time step
        slots_past = []
        for t in range(3):
            x_t = self.in_proj(feats[:, t])      # (B,N,d_token)
            s_t = self.slot_attn(x_t)            # (B,K,d_slot)
            slots_past.append(s_t)
        slots_past = torch.stack(slots_past, dim=1)  # (B,3,K,d_slot)

        state6 = self.state_enc(robot_states_34x25)   # (B,6,d_state)

        slots_future = self.dynamics(slots_past, state6)  # (B,3,K,d_slot)

        # decode each future time step to vocab logits over grid
        logits_list = []
        for t in range(3):
            logits_t = self.decoder(slots_future[:, t])   # (B,N,V)
            logits_list.append(logits_t)
        logits = torch.stack(logits_list, dim=1)          # (B,3,N,V)
        return logits


In [ ]:
from torch.utils.data import DataLoader
import wandb

def collate_fn(batch):
    # batch: list of dicts
    past = torch.stack([torch.from_numpy(b["past_frames"]) for b in batch], dim=0).long()      # (B,3,32,32)
    fut  = torch.stack([torch.from_numpy(b["future_frames"]) for b in batch], dim=0).long()    # (B,3,32,32)
    st   = torch.stack([torch.from_numpy(b["robot_states"]) for b in batch], dim=0).float()    # (B,34,25)
    return {"past": past, "future": fut, "states": st}

import time
import torch
import torch.nn.functional as F
from torch import nn

def train_one_epoch_wandb_amp(
    model,
    loader,
    optimizer,
    device="cuda",
    grad_clip=1.0,
    log_every=50,
    epoch=0,
    global_step=0,
    amp_dtype="bf16",   # "bf16" or "fp16"
    scaler=None,        # 外から渡す（推奨）
):
    assert device.startswith("cuda"), "AMPはCUDA前提"
    model.train()

    # GradScaler: fp16では必須、bf16では基本不要だが、統一のため使ってOK
    if scaler is None:
        scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == "fp16"))

    # autocast dtype
    autocast_dtype = torch.bfloat16 if amp_dtype == "bf16" else torch.float16
    autocast_enabled = True

    total_loss = 0.0
    total_tokens = 0
    ema = None
    ema_beta = 0.98
    t0 = time.time()

    pbar = tqdm(loader, desc=f"train epoch {epoch}", leave=False)

    for step, batch in enumerate(pbar):
        past = batch["past"].to(device, non_blocking=True)
        future = batch["future"].to(device, non_blocking=True)
        states = batch["states"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # ★ここがAMPを挟む場所（forward〜loss）
        with torch.cuda.amp.autocast(enabled=autocast_enabled, dtype=autocast_dtype):
            logits = model(past, states)  # (B,3,N,V)
            B, T, N, V = logits.shape
            target = future.view(B, T, -1)  # (B,3,1024)

            loss = F.cross_entropy(
                logits.reshape(B * T * N, V),
                target.reshape(B * T * N),
                reduction="mean",
            )

        # ★backward/stepはscaler経由（fp16のとき安全）
        if amp_dtype == "fp16":
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            # bf16ならscaler無しでOK（多くの環境で高速・安定）
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        tokens = B * T * N
        loss_val = float(loss.item())
        total_loss += loss_val * tokens
        total_tokens += tokens
        ema = loss_val if ema is None else (ema_beta * ema + (1 - ema_beta) * loss_val)

        dt = time.time() - t0
        tok_per_s = total_tokens / max(dt, 1e-9)
        lr = optimizer.param_groups[0]["lr"]

        wandb.log(
            {
                "train/loss_step": loss_val,
                "train/loss_ema": float(ema),
                "train/lr": lr,
                "train/tokens_seen": total_tokens,
                "train/tok_per_sec": tok_per_s,
                "epoch": epoch,
                "amp/dtype": amp_dtype,
            },
            step=global_step,
        )

        pbar.set_postfix(
            loss=f"{loss_val:.4f}",
            ema=f"{ema:.4f}",
            lr=f"{lr:.2e}",
            tok_s=f"{tok_per_s:,.0f}",
        )

        global_step += 1

    epoch_loss = total_loss / max(1, total_tokens)
    wandb.log({"train/loss_epoch": epoch_loss}, step=global_step)
    return epoch_loss, global_step, scaler

In [ ]:
device = "cuda"
root = "/root/work/data/raw/train_v2.0"

from dataclasses import asdict

# ---- single source of truth ----
cfg = ModelConfig(
    vocab_size=65536,
    num_slots=8,
    d_token=256,
    d_slot=128,
    d_state=128,
    slot_iters=3,
    H=32, W=32
)

train_cfg = {
    "batch_size": 8,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "log_every": 10,
    "epochs": 1,
    "num_workers": 4,
}


wandb.init(
    project="slot-attn-token-pred",
    name="run2",
    config={**asdict(cfg), **train_cfg},  # cfg をそのまま wandb に同期
)


model = SlotPredictor(cfg).to(device)

ds = TrainDataset(
    root="/root/work/data/raw/train_v2.0",
    output_format="seq2seq",
    cache_path="/root/work/data/outputs/valid_starts_stride3_clipclean.npy",
)

loader = DataLoader(
    ds,
    batch_size=train_cfg["batch_size"],
    shuffle=True,
    num_workers=train_cfg["num_workers"],
    pin_memory=True,
    collate_fn=collate_fn,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg["lr"],
    weight_decay=train_cfg["weight_decay"],
)
scaler = None
for epoch in range(train_cfg["epochs"]):
    avg, global_step, scaler = train_one_epoch_wandb_amp(
        model, loader, optimizer,
        device="cuda",
        grad_clip=train_cfg["grad_clip"],
        log_every=train_cfg["log_every"],
        epoch=epoch,
        global_step=global_step,
        amp_dtype="bf16",  # bf16が使えるGPUならこれ
        scaler=scaler
    )

wandb.finish()


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁
train/loss_ema,█▁
train/loss_step,█▁
train/lr,▁▁
train/tok_per_sec,▁█
train/tokens_seen,▁█
amp/dtype,bf16
epoch,0
train/loss_ema,11.25276
train/loss_step,11.24232
train/lr,0.0002


/tmp/ipykernel_89406/1354236964.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(amp_dtype == "fp16"))
train epoch 0:   0%|          | 0/77454 [00:00<?, ?it/s]/tmp/ipykernel_89406/1354236964.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=autocast_enabled, dtype=autocast_dtype):
train epoch 0:   0%|          | 5/77454 [01:00<262:17:31, 12.19s/it, ema=11.2504, loss=11.2186, lr=2.00e-04, tok_s=2,032]